# 10 · Revisión de `04` §8 y §15 con el bootstrap que repite el ajuste

En `04` hay dos familias de comparaciones cuyos valores p salieron de un bootstrap que deja fijas las predicciones de la validación cruzada. Ese bootstrap da p demasiado chicos (`docs/criterio_parsimonia.md`, agregado del 2026-11-20). Acá se recalculan con el **detector final** y con el bootstrap que repite toda la validación cruzada en cada sorteo de personas. El plan está en `docs/revision_04.md`, escrito y subido antes de correr este notebook:

1. **Decisiones de diseño** (`04` §15): 13 variantes, en exactitud.
2. **Otras combinaciones de RR y EDR** (`04` §8): C contra 9 alternativas, en 7 medidas.

No decide nada: las decisiones de `04` no cambian. Si una alternativa resultara mejor de forma demostrable, se informa y se consulta. Test no se usa.

In [1]:
import json
import os
import sys
import itertools
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import src.features.detector as det_mod
from src.config import MIN_MINUTES_PER_CLASS, SPLIT_CSV
from src.estadistica import holm, p_bootstrap
from src.evaluation.cv import GRILLA_W, auc, cv_c, grupos_personas, metricas, sorteos_personas
from src.features.build import record_series
from src.features.detector import WINDOW_S, c_score, evaluable, fit_c, youden_threshold

split = pd.read_csv(SPLIT_CSV)
DEV = split[split.set == "dev"].record.tolist()
RAPIDO = bool(os.environ.get("NB10_RAPIDO"))   # solo para probar el código: 16 registros y pocos sorteos
if RAPIDO:
    DEV = DEV[:10] + ["c01", "c09", "x03", "x17", "x28", "x35"]
N_SORTEOS = 20 if RAPIDO else 2000
SUJETO = split.set_index("record").subject
CATEGORIA = split.set_index("record").category
PERSONAS = sorted(SUJETO[DEV].unique())
SORTEOS = sorteos_personas(PERSONAS, n=N_SORTEOS, semilla=2026)

PARAMS = json.loads((ROOT / "docs" / "parametros_final.json").read_text())
ET = PARAMS["etapas"]
FINAL_EDR = {"edr_method": "area", "wiener": ET["wiener_edr"], "exclude_ectopic": ET["excluir_ectopicos_edr"],
             "hampel": ET["hampel_edr"], "aberrant": ET["aberrantes_varon_edr"], "relative": ET["edr_relativa"],
             "interp": ET["interpolacion"]}
DT, MC = ET["tendencia_welch"], ET["max_rr_corregidos_por_minuto"]


def replicar(e, muestra):
    """Minutos de las personas sorteadas; cada copia de una noche tiene su propio identificador (rid)."""
    s = e.subject.to_numpy()
    partes, vistas = [], {}
    for u in muestra:
        k = vistas.get(u, 0); vistas[u] = k + 1
        parte = e.iloc[np.flatnonzero(s == u)].copy()
        parte["rid"] = parte.record + f"#{k}"
        partes.append(parte)
    return pd.concat(partes, ignore_index=True)


def con_rid(e):
    return e.assign(rid=e.record + "#0")


def justas(e, extra=()):
    """Solo las columnas que usa el ajuste: filtrar tablas anchas en cada sorteo es lo que más tarda."""
    return e[["record", "subject", "minute", "label", "P_RR", "EDR_noche", *extra]].reset_index(drop=True)

## 1. Decisiones de diseño (`04` §15)

Las mismas 13 variantes de `04` §15, ahora sobre el detector final. Cada variante cambia una decisión y se repite toda la validación cruzada. La exactitud se compara en los minutos que las dos versiones evalúan.

In [2]:
def tabla_registro(args):
    rec, op = args
    det_mod.WINDOW_S = op.get("ventana", WINDOW_S)
    kw = {**FINAL_EDR, **({"interp": "linear"} if op.get("interp") == "lineal" else {})}
    t = det_mod.minute_table(rec, series=record_series(rec, **kw), centered=op.get("centered", True),
                             band_edr=op.get("band_edr"), detrend_type=op.get("detrend", DT))
    det_mod.WINDOW_S = WINDOW_S
    return t


def tablas(op):
    with ProcessPoolExecutor(4) as ex:
        t = pd.concat(list(ex.map(tabla_registro, [(r, op) for r in DEV])), ignore_index=True)
    return t.assign(subject=t.record.map(SUJETO))


def evaluables(T_, nivel="median", escala="log", sin_primera_hora=False):
    T_ = T_.copy()
    if escala == "lineal":
        T_ = T_.assign(P_RR=10 ** T_.P_RR, P_EDR=10 ** T_.P_EDR)
    if escala == "lineal" or nivel != "median":
        T_["EDR_noche"] = T_.groupby("record").P_EDR.transform(nivel)
    e = T_[evaluable(T_, MC)]
    if sin_primera_hora:
        e = e[e.minute >= 60]
    return e[np.isfinite(e.P_RR) & np.isfinite(e.EDR_noche)].reset_index(drop=True)


VARIANTES_T = {
    "actual": {}, "interpolación lineal": {"interp": "lineal"},
    "ventana de 180 s": {"ventana": 180}, "ventana de 300 s": {"ventana": 300}, "ventana de 360 s": {"ventana": 360},
    "sin quitar la tendencia": {"detrend": "none"}, "quitando solo la media": {"detrend": "constant"},
    "ventana hacia atrás (solo pasado)": {"centered": False}, "EDR en 0,017–0,033 Hz": {"band_edr": (0.01, 0.034)},
}
T_T = {nombre: tablas(op) for nombre, op in VARIANTES_T.items()}
E_ACT = evaluables(T_T["actual"])
VARIANTES = {nombre: (justas(evaluables(t)), 5) for nombre, t in T_T.items() if nombre != "actual"}
VARIANTES.update({
    "potencias en escala lineal": (justas(evaluables(T_T["actual"], escala="lineal")), 5),
    "nivel nocturno = media": (justas(evaluables(T_T["actual"], nivel="mean")), 5),
    "10 grupos": (justas(E_ACT), 10), "una persona por grupo": (justas(E_ACT), len(PERSONAS)),
    "sin la primera hora": (justas(evaluables(T_T["actual"], sin_primera_hora=True)), 5),
})
print("el actual es el detector final:", bool(np.isclose(
    youden_threshold(c_score(E_ACT, fit_c(E_ACT, PARAMS["w"], max_corrected=None)).to_numpy(), E_ACT.label.to_numpy()), PARAMS["umbral"])))

el actual es el detector final: True


In [3]:
E_ACT_J = justas(E_ACT)


def aciertos(e, n_grupos):
    """Validación cruzada de una versión: si acierta en cada minuto (rid, minute)."""
    r = cv_c(e, n_grupos=n_grupos)
    return pd.DataFrame({"rid": r.rid, "minute": r.minute, "ok": (r.pred == r.label).astype(float)})


def dif_comun(ra, rb):
    """Exactitud de a menos la de b en los minutos que las dos evalúan."""
    m_ = ra.merge(rb, on=["rid", "minute"], suffixes=("_a", "_b"))
    return float(m_.ok_a.mean() - m_.ok_b.mean())


def diferencias_t(muestra=None):
    """Las 13 diferencias con el actual en un sorteo (o con todas las personas si muestra es None).
    La validación cruzada del actual se hace una sola vez por sorteo."""
    prep = con_rid if muestra is None else (lambda e: replicar(e, muestra))
    r_act = aciertos(prep(E_ACT_J), 5)
    return {n: dif_comun(aciertos(prep(e_v), g_v), r_act) for n, (e_v, g_v) in VARIANTES.items()}


def _trabajo_t(idx):
    return [diferencias_t(SORTEOS[i]) for i in idx]


PUNTO_T = diferencias_t()
with ProcessPoolExecutor(4) as ex:
    BOOT_T = [x for b in ex.map(_trabajo_t, np.array_split(np.arange(N_SORTEOS), 40)) for x in b]
RES_T = {n: (PUNTO_T[n], np.array([b_[n] for b_ in BOOT_T])) for n in VARIANTES}
PH_T = holm({k: p_bootstrap(b) for k, (_, b) in RES_T.items()})
TAB_T = pd.DataFrame({"variante − actual": {k: v[0] for k, v in RES_T.items()},
                      "IC 95 %": {k: f"{np.nanpercentile(v[1], 2.5):+.3f} a {np.nanpercentile(v[1], 97.5):+.3f}" for k, v in RES_T.items()},
                      "p": {k: p_bootstrap(v[1]) for k, v in RES_T.items()}, "p de Holm": PH_T})
TAB_T["p de Holm"] = TAB_T["p de Holm"].fillna(1.0)
TAB_T["diferencia demostrable"] = np.where(TAB_T["p de Holm"] < 0.05, "sí", "no")
TAB_T.sort_values("p").round(4)

,variante − actual,IC 95 %,p,p de Holm,diferencia demostrable
potencias en escala lineal,-0.0650,-0.120 a +0.013,0.086,1.0,no
ventana de 360 s,-0.0016,-0.027 a +0.018,0.278,1.0,no
nivel nocturno = media,-0.0095,-0.038 a +0.020,0.316,1.0,no
ventana de 300 s,0.0033,-0.020 a +0.019,0.362,1.0,no
interpolación lineal,0.0171,-0.011 a +0.019,0.390,1.0,no
ventana hacia atrás (solo pasado),0.0024,-0.018 a +0.013,0.404,1.0,no
ventana de 180 s,0.0065,-0.017 a +0.019,0.687,1.0,no
una persona por grupo,0.0159,-0.025 a +0.047,0.759,1.0,no
"EDR en 0,017–0,033 Hz",0.0110,-0.014 a +0.026,0.780,1.0,no
quitando solo la media,0.0136,-0.008 a +0.008,0.781,1.0,no


**Por qué algunas estimaciones quedan en el borde de su intervalo.** La estimación es una sola validación cruzada, con un reparto fijo de las personas en grupos. Si en un grupo el peso w elegido cambia de un escalón de la grilla al siguiente, la exactitud de ese grupo puede saltar varios puntos. Para las variantes con 5 grupos (los mismos grupos que el actual), la tabla muestra los grupos donde la variante elige otro w y cuánto aporta ese grupo a la diferencia.

In [4]:
R_ACT0 = cv_c(con_rid(E_ACT_J))
filas = []
for n, (e_v, g_v) in VARIANTES.items():
    if g_v != 5:
        continue
    m_ = cv_c(con_rid(e_v)).merge(R_ACT0, on=["rid", "minute"], suffixes=("_v", "_a"))
    m_["dif"] = (m_.pred_v == m_.label_v).astype(float) - (m_.pred_a == m_.label_a).astype(float)
    for k, d_ in m_.groupby("grupo_a"):
        if d_.w_v.iloc[0] != d_.w_a.iloc[0]:
            filas.append({"variante": n, "grupo": int(k), "w actual": d_.w_a.iloc[0], "w variante": d_.w_v.iloc[0],
                          "minutos del grupo": len(d_), "diferencia en el grupo": d_.dif.mean(),
                          "diferencia total": m_.dif.mean(), "diferencia sin ese grupo": m_.dif[m_.grupo_a != k].mean()})
pd.DataFrame(filas).set_index(["variante", "grupo"]).round(4)

w actual  w variante  \
variante                          grupo                         
interpolación lineal              1          0.50        0.75   
ventana de 180 s                  0          1.25        1.50   
                                  1          0.50        0.75   
ventana de 300 s                  0          1.25        1.50   
                                  1          0.50        0.75   
ventana de 360 s                  0          1.25        1.50   
                                  1          0.50        0.75   
                                  3          1.00        1.25   
sin quitar la tendencia           1          0.50        0.75   
quitando solo la media            1          0.50        0.75   
ventana hacia atrás (solo pasado) 0          1.25        1.50   
                                  1          0.50        0.75   
EDR en 0,017–0,033 Hz             0          1.25        1.50   
                                  1          0.50        0.75   
potencias en escala lineal        0          1.25        1.75   
                                  1          0.50        0.25   
                                  2          1.00        0.50   
                                  4          1.00        0.50   
nivel nocturno = media            4          1.00        0.75   
sin la primera hora               0          1.25        1.50   
                                  1          0.50        0.75   

                                         minutos del grupo  \
variante                          grupo                      
interpolación lineal              1                   3852   
ventana de 180 s                  0                   4979   
                                  1                   3852   
ventana de 300 s                  0                   4968   
                                  1                   3844   
ventana de 360 s                  0                   4964   
                                  1                   3841   
                                  3                   4375   
sin quitar la tendencia           1                   3852   
quitando solo la media            1                   3852   
ventana hacia atrás (solo pasado) 0                   4963   
                                  1                   3839   
EDR en 0,017–0,033 Hz             0                   4979   
                                  1                   3852   
potencias en escala lineal        0                   4979   
                                  1                   3852   
                                  2                   5156   
                                  4                   4393   
nivel nocturno = media            4                   4393   
sin la primera hora               0                   4419   
                                  1                   3399   

                                         diferencia en el grupo  \
variante                          grupo                           
interpolación lineal              1                      0.0857   
ventana de 180 s                  0                     -0.0121   
                                  1                      0.0652   
ventana de 300 s                  0                     -0.0107   
                                  1                      0.0715   
ventana de 360 s                  0                     -0.0091   
                                  1                      0.0685   
                                  3                     -0.0247   
sin quitar la tendencia           1                      0.0753   
quitando solo la media            1                      0.0753   
ventana hacia atrás (solo pasado) 0                     -0.0115   
                                  1                      0.0646   
EDR en 0,017–0,033 Hz             0                     -0.0080   
                                  1                      0.0706   
potencias en escala lineal        0           

**Qué se ve.**
- **Ninguna de las 13 variantes cambia la exactitud de forma demostrable.** El p más chico es 0,086 y con Holm todos quedan en 1. En `04` §15 resistían Holm dos diferencias, y con este bootstrap ya no:
  - la ventana de 360 s da −0,0016 (p = 0,28);
  - la escala lineal da −0,065 (IC −0,120 a +0,013; p = 0,086).
- **La escala lineal es la única con una diferencia grande y en una sola dirección.** Empeora en los cuatro grupos donde elige otro w. Sin cada uno de esos grupos sigue entre −0,068 y −0,072. Va en el sentido de la razón física para usar el logaritmo, pero con 19 personas no llega a ser demostrable.
- **"Quitando solo la media" y "sin quitar la tendencia" dan exactamente lo mismo, y no es un error.** Con la ventana de Hann, una constante solo deja potencia en los bins k = 0 y 1. La banda usa k = 2 a 6, así que quitar la media no cambia ninguna de sus potencias.
- **Varias variantes salen entre +0,011 y +0,017 con un intervalo que casi no las contiene**, y en dos ("quitando solo la media" y "sin quitar la tendencia") la estimación queda afuera. Casi todo viene del grupo 1:
  - ahí el detector final elige w = 0,50 y esas variantes eligen 0,75;
  - ese salto de un escalón de la grilla da entre 6,5 y 8,6 puntos de exactitud en ese grupo;
  - sin ese grupo, las diferencias quedan entre −0,016 y +0,003.

  Es una propiedad de este reparto de las personas en grupos, no de las variantes. En los sorteos del bootstrap ese salto casi no se repite, y por eso el intervalo queda corrido hacia cero. No hay que leer esas estimaciones como mejoras.

## 2. Otras combinaciones de RR y EDR (`04` §8)

C contra las mismas alternativas de `04` §8, todas sobre los mismos minutos del detector final. En cada grupo de la validación cruzada se estandarizan los términos, se eligen los pesos que maximizan el AUC y se fija el umbral de Youden; la regresión logística ajusta sus coeficientes. Las 7 medidas son las de `04`. En el bootstrap se repite todo eso en cada sorteo, y cada copia de una noche cuenta como una noche aparte.

In [5]:
G_PESO = GRILLA_W; G_CHICA = np.arange(0, 1.51, 0.25)
CANDIDATAS = {
    "C: RR + w·nivel EDR": ("P_RR", [("EDR_noche", G_PESO)]),
    "RR solo": ("P_RR", []), "EDR sola": ("P_EDR", []), "F (potencia cruzada)": ("F", []),
    "RR + EDR (minuto), pesos iguales": ("P_RR", [("P_EDR", [1.0])]),
    "RR + w·EDR (minuto)": ("P_RR", [("P_EDR", G_PESO)]),
    "G: RR centrado + w·nivel EDR": ("RR_dev", [("EDR_noche", G_PESO)]),
    "RR + w₁·nivel EDR + w₂·desvío EDR": ("P_RR", [("EDR_noche", G_PESO), ("EDR_dev", G_CHICA)]),
    "RR centrado + w₁·nivel RR + w₂·nivel EDR": ("RR_dev", [("RR_noche", G_CHICA), ("EDR_noche", G_PESO)]),
    "Regresión logística, 4 términos (referencia)": (None, None),
}
NOMBRE_C = "C: RR + w·nivel EDR"
E8 = justas(E_ACT.assign(RR_noche=E_ACT.P_RR - E_ACT.RR_dev, EDR_dev=E_ACT.P_EDR - E_ACT.EDR_noche),
           ["P_EDR", "F", "RR_dev", "RR_noche", "EDR_dev"])


def _par(d, cols):
    return {c: (d[c].mean(), d[c].std()) for c in cols}


def _suma(d, base, pesos, par):
    s = (d[base].to_numpy() - par[base][0]) / par[base][1]
    for c, w in pesos:
        s = s + w * (d[c].to_numpy() - par[c][0]) / par[c][1]
    return s


def _logit(X, yy, it=40):
    X = np.c_[np.ones(len(X)), X]; b = np.zeros(X.shape[1])
    for _ in range(it):
        p = 1 / (1 + np.exp(-X @ b)); H = X.T @ (X * (p * (1 - p))[:, None]) + 1e-6 * np.eye(len(b))
        b += np.linalg.solve(H, X.T @ (yy - p))
    return b


def cv_candidata(e, base, libres):
    """Validación cruzada por personas de una candidata (como en 04 §8): predicción, margen y grupo de cada minuto."""
    y = e.label.to_numpy(); s = e.subject.to_numpy()
    pred = np.zeros(len(e), int); marg = np.zeros(len(e)); grupo = np.zeros(len(e), int)
    for k, gpo in enumerate(grupos_personas(s)):
        te = np.isin(s, gpo); tr = ~te; dtr, dte = e[tr], e[te]
        if base is None:
            cols = ["RR_dev", "RR_noche", "EDR_dev", "EDR_noche"]; par = _par(dtr, cols)
            Z = lambda d: np.column_stack([(d[c] - par[c][0]) / par[c][1] for c in cols])
            b = _logit(Z(dtr), y[tr])
            s_tr, s_te = np.c_[np.ones(tr.sum()), Z(dtr)] @ b, np.c_[np.ones(te.sum()), Z(dte)] @ b
        else:
            par = _par(dtr, [base] + [c for c, _ in libres]); mejor = (-1, [])
            for combo in itertools.product(*[g for _, g in libres]):
                pesos = list(zip([c for c, _ in libres], combo)); a_ = auc(_suma(dtr, base, pesos, par), y[tr])
                if a_ > mejor[0]:
                    mejor = (a_, pesos)
            s_tr, s_te = _suma(dtr, base, mejor[1], par), _suma(dte, base, mejor[1], par)
        u_ = youden_threshold(s_tr, y[tr]); pred[te] = s_te > u_; marg[te] = s_te - u_; grupo[te] = k
    return pred, marg, grupo


def medidas(e, pred, marg, grupo):
    y = e.label.to_numpy()
    aucs = [auc(marg[grupo == k], y[grupo == k]) for k in np.unique(grupo) if 0 < y[grupo == k].sum() < (grupo == k).sum()]
    d = pd.DataFrame({"rid": e.rid.to_numpy(), "record": e.record.to_numpy(), "y": y, "p": pred, "m": marg})
    noches = d.groupby("rid").agg(real=("y", "mean"), est=("p", "mean"), record=("record", "first"))
    dentro = [auc(g_.m.to_numpy(), g_.y.to_numpy()) for _, g_ in d.groupby("rid")
              if min((g_.y == 1).sum(), (g_.y == 0).sum()) >= MIN_MINUTES_PER_CLASS]
    ctrl = noches[noches.record.map(CATEGORIA) == "C"]
    return {**metricas(y, pred), "AUC (personas no vistas)": float(np.mean(aucs)),
            "AUC dentro de la noche": float(np.mean(dentro)), "ρ por noche": spearmanr(noches.real, noches.est)[0],
            "% marcado en controles": 100 * ctrl.est.mean() if len(ctrl) else np.nan}


def todas(e):
    return {n: medidas(e, *cv_candidata(e, *spec)) for n, spec in CANDIDATAS.items()}


def _trabajo_8(idx):
    return [todas(replicar(E8, SORTEOS[i])) for i in idx]


PUNTO = todas(con_rid(E8))
with ProcessPoolExecutor(4) as ex:
    BOOT8 = [x for b in ex.map(_trabajo_8, np.array_split(np.arange(N_SORTEOS), 40)) for x in b]
pd.DataFrame(PUNTO).T.round(3)

,sensibilidad,especificidad,exactitud,AUC (personas no vistas),AUC dentro de la noche,ρ por noche,% marcado en controles
C: RR + w·nivel EDR,0.780,0.863,0.830,0.888,0.908,0.739,9.539
RR solo,0.748,0.734,0.740,0.824,0.908,0.359,36.860
EDR sola,0.820,0.722,0.761,0.852,0.797,0.773,15.489
F (potencia cruzada),0.842,0.787,0.809,0.900,0.877,0.847,16.037
"RR + EDR (minuto), pesos iguales",0.848,0.785,0.810,0.893,0.886,0.804,19.035
RR + w·EDR (minuto),0.794,0.774,0.782,0.863,0.877,0.712,20.800
G: RR centrado + w·nivel EDR,0.828,0.729,0.769,0.912,0.908,0.708,14.342
RR + w₁·nivel EDR + w₂·desvío EDR,0.798,0.838,0.822,0.885,0.899,0.730,10.151
RR centrado + w₁·nivel RR + w₂·nivel EDR,0.804,0.832,0.821,0.898,0.908,0.751,10.759
"Regresión logística, 4 términos (referencia)",0.806,0.827,0.818,0.891,0.907,0.729,11.969


In [6]:
MEDIDAS = list(PUNTO[NOMBRE_C])
RES_8 = {}
for n in CANDIDATAS:
    if n == NOMBRE_C:
        continue
    for m in MEDIDAS:
        b = np.array([s_[NOMBRE_C][m] - s_[n][m] for s_ in BOOT8], float)
        RES_8[(n, m)] = (PUNTO[NOMBRE_C][m] - PUNTO[n][m], b)
P_8 = {k: p_bootstrap(b) for k, (_, b) in RES_8.items()}
PH_8 = holm(P_8)
TAB_8 = pd.DataFrame([{"C contra": k[0], "medida": k[1], "C − candidata": d,
                       "IC 95 %": f"{np.nanpercentile(b, 2.5):+.3f} a {np.nanpercentile(b, 97.5):+.3f}",
                       "p": P_8[k], "p de Holm": PH_8.get(k, np.nan)} for k, (d, b) in RES_8.items()]).set_index(["C contra", "medida"])
print(f"{TAB_8.p.notna().sum()} diferencias con p · significativas sin corregir: {(TAB_8.p < 0.05).sum()} · con Holm: {(TAB_8['p de Holm'] < 0.05).sum()}")
TAB_8[TAB_8.p < 0.05].sort_values("p").round(4)

60 diferencias con p · significativas sin corregir: 7 · con Holm: 0


C − candidata  \
C contra                         medida                                  
EDR sola                         AUC dentro de la noche         0.1114   
RR + w·EDR (minuto)              AUC dentro de la noche         0.0315   
F (potencia cruzada)             AUC dentro de la noche         0.0305   
RR solo                          % marcado en controles       -27.3203   
                                 ρ por noche                    0.3801   
RR + w·EDR (minuto)              % marcado en controles       -11.2608   
RR + EDR (minuto), pesos iguales AUC dentro de la noche         0.0224   

                                                                   IC 95 %  \
C contra                         medida                                      
EDR sola                         AUC dentro de la noche    +0.061 a +0.159   
RR + w·EDR (minuto)              AUC dentro de la noche    +0.013 a +0.053   
F (potencia cruzada)             AUC dentro de la noche    +0.011 a +0.042   
RR solo                          % marcado en controles  -54.302 a -10.003   
                                 ρ por noche               +0.060 a +0.899   
RR + w·EDR (minuto)              % marcado en controles   -18.216 a -2.319   
RR + EDR (minuto), pesos iguales AUC dentro de la noche    +0.001 a +0.037   

                                                              p  p de Holm  
C contra                         medida                                     
EDR sola                         AUC dentro de la noche  0.0010      0.060  
RR + w·EDR (minuto)              AUC dentro de la noche  0.0030      0.177  
F (potencia cruzada)             AUC dentro de la noche  0.0050      0.290  
RR solo                          % marcado en controles  0.0050      0.290  
                                 ρ por noche             0.0190      1.000  
RR + w·EDR (minuto)              % marcado en controles  0.0221      1.000  
RR + EDR (minuto), pesos iguales AUC dentro de la noche  0.0420      1.000

Para el % marcado en controles, una diferencia negativa favorece a C (marca menos minutos en noches sin apnea); para las demás medidas, una positiva. La tabla completa, con la diferencia C − candidata y, entre paréntesis, su p de Holm:

In [7]:
TAB_8.apply(lambda f: f"{f['C − candidata']:+.3f} ({f['p de Holm']:.2f})" if np.isfinite(f["p de Holm"])
            else f"{f['C − candidata']:+.3f} (—)", axis=1).unstack("medida").reindex([n for n in CANDIDATAS if n != NOMBRE_C])[MEDIDAS]

medida,sensibilidad,especificidad,exactitud,AUC (personas no vistas),AUC dentro de la noche,ρ por noche,% marcado en controles
C contra,,,,,,,
RR solo,+0.032 (1.00),+0.129 (1.00),+0.090 (1.00),+0.063 (1.00),+0.000 (—),+0.380 (1.00),-27.320 (0.29)
EDR sola,-0.040 (1.00),+0.141 (1.00),+0.069 (1.00),+0.036 (1.00),+0.111 (0.06),-0.034 (1.00),-5.950 (1.00)
F (potencia cruzada),-0.062 (1.00),+0.076 (1.00),+0.021 (1.00),-0.013 (1.00),+0.031 (0.29),-0.108 (1.00),-6.497 (1.00)
"RR + EDR (minuto), pesos iguales",-0.067 (1.00),+0.078 (1.00),+0.020 (1.00),-0.005 (1.00),+0.022 (1.00),-0.065 (1.00),-9.496 (1.00)
RR + w·EDR (minuto),-0.013 (1.00),+0.090 (1.00),+0.048 (1.00),+0.025 (1.00),+0.031 (0.18),+0.027 (1.00),-11.261 (1.00)
G: RR centrado + w·nivel EDR,-0.048 (1.00),+0.135 (1.00),+0.061 (1.00),-0.025 (1.00),+0.000 (—),+0.031 (1.00),-4.803 (1.00)
RR + w₁·nivel EDR + w₂·desvío EDR,-0.017 (1.00),+0.025 (1.00),+0.008 (1.00),+0.003 (1.00),+0.009 (1.00),+0.009 (1.00),-0.611 (1.00)
RR centrado + w₁·nivel RR + w₂·nivel EDR,-0.024 (1.00),+0.031 (1.00),+0.009 (1.00),-0.011 (1.00),+0.000 (—),-0.012 (1.00),-1.220 (1.00)
"Regresión logística, 4 términos (referencia)",-0.025 (1.00),+0.037 (1.00),+0.012 (1.00),-0.004 (1.00),+0.001 (1.00),+0.010 (1.00),-2.429 (1.00)


**Qué se ve.**
- **Ninguna de las 60 diferencias resiste Holm.** En `04` resistían 4.
- **Sin corregir son significativas 7, y las 7 favorecen a C:**
  - dentro de la noche, contra la EDR sola (+0,111), contra F (+0,031), contra RR + w·EDR por minuto (+0,031) y contra la suma con pesos iguales (+0,022);
  - en los controles, contra el RR solo (−27,3 puntos de % marcado) y contra RR + w·EDR por minuto (−11,3);
  - en ρ por noche, contra el RR solo (+0,380).
- **Ninguna alternativa es mejor que C de forma demostrable en ninguna medida, ni siquiera sin corregir.** Algunas lo superan en la estimación, sin que la diferencia sea demostrable:
  - en AUC, G (0,912) y F (0,900) contra 0,888;
  - en ρ por noche, F (0,847) contra 0,739;
  - en sensibilidad, casi todas.

  C tiene la mayor exactitud (0,830), la mayor especificidad (0,863) y el menor % marcado en los controles (9,5 %).
- **Contra el RR solo, G y "RR centrado + w₁·nivel RR + w₂·nivel EDR", el AUC dentro de la noche es idéntico** (+0,000, sin p). Esas combinaciones difieren de C en términos que son constantes en cada noche, y una constante no cambia el orden de los minutos dentro de la noche.

## 3. Conclusión

- **Con el detector final y el bootstrap que repite el ajuste, ninguna comparación de `04` §8 y §15 resiste Holm.** Las afirmaciones de `04` que decían lo contrario (4 diferencias en §8, 2 en §15) se corrigieron con una nota fechada en `04`, `decisiones.md`, `auditoria_decisiones.md` y `objetivos_hipotesis.md`. Los resultados de `04` con el bootstrap anterior quedan como estaban.
- **Ninguna alternativa ni variante es mejor que el detector final de forma demostrable, ni siquiera sin corregir.** Según el plan (`revision_04.md`), no hay nada que consultar y las decisiones de `04` no cambian. Se sostienen por su razón física o de método, no por estas diferencias.
- **Lo que ya no se puede afirmar con desarrollo:**
  - que C marca menos minutos que el RR solo en los controles (−27,3 puntos, p de Holm = 0,29);
  - que la escala lineal es peor que el logaritmo;
  - que la ventana de 360 s es peor que la de 240 s.

  En test la versión evaluada de C marcó menos minutos en los controles que el RR solo (−22,9 puntos) y que F (−10,4), y esas diferencias resisten Holm (`05`, `06` §5). Ahí el detector estaba fijo, así que corresponde el bootstrap con las predicciones fijas.